# EEG feature extraction and exploratory quality checks

## Purpose

This notebook converts each balanced EEG epoch file produced by the preceding
epoch-extraction stage into a numerical feature matrix for machine-learning
analysis.

For every 2-second epoch, the script:

1. Retains EEG channels only.
2. Averages the EEG signal across channels to produce one representative
   time series per epoch.
3. Extracts five time-domain features: mean, standard deviation, maximum,
   minimum and root mean square (RMS).
4. Estimates the power spectral density using Welch's method.
5. Calculates absolute power in the delta, theta, alpha and beta bands.
6. Saves the resulting feature matrix and class labels in a compressed
   NumPy (`.npz`) file.

The second half of the notebook contains exploratory checks used to confirm
that the saved files are complete, contain finite values and show plausible
differences between SWD and non-SWD epochs. PCA is used only for visualisation;
its output is not used to train the classifier.

## Expected inputs

- MNE epoch files in `epochs_2/`
- Filenames ending in `-edf_chin_epochs-epo.fif`
- Event labels encoded as `1 = SWD` and `0 = non-SWD`

## Outputs

One `.npz` file per input recording is written to `features/`, containing:

- `X`: feature matrix with one row per epoch
- `y`: class label for each epoch
- `feature_names`: names corresponding to the columns of `X`

Directory paths are relative to the working directory. The notebook should
therefore be run from the project directory containing `epochs_2/`.


## 1. Extract temporal and power features


In [ ]:

import os
import glob

import numpy as np
import mne
from mne.time_frequency import psd_array_welch


# -------------------------------------------------------------------------
# Input and output directories
# -------------------------------------------------------------------------

# Balanced 2-second epoch files generated by the preceding pipeline stage.
EPOCHS_DIR = "epochs_2"

# One compressed feature file will be written here for each epoch file.
FEATURES_DIR = "features"
os.makedirs(FEATURES_DIR, exist_ok=True)


# -------------------------------------------------------------------------
# Frequency bands used for spectral feature extraction
# -------------------------------------------------------------------------

# Band limits are expressed in hertz. Power is calculated separately within
# each interval after estimating the power spectral density from 1 to 40 Hz.
bands = {
    "delta_1_4": (1, 4),
    "theta_4_8": (4, 8),
    "alpha_8_13": (8, 13),
    "beta_13_30": (13, 30),
}


# -------------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------------

def rms(x):
    """Return the root mean square amplitude of a one-dimensional signal."""
    return np.sqrt(np.mean(x ** 2))


def band_power(psd, freqs, fmin, fmax):
    """
    Calculate absolute power within a specified frequency interval.

    Parameters
    ----------
    psd : numpy.ndarray
        Power spectral density values.
    freqs : numpy.ndarray
        Frequency value corresponding to each PSD estimate.
    fmin, fmax : float
        Lower and upper limits of the frequency band in hertz.

    Returns
    -------
    float
        Area under the PSD curve within the selected frequency band.
    """
    # Select only the PSD estimates that fall inside the requested band.
    mask = (freqs >= fmin) & (freqs <= fmax)

    # Integrating the PSD across frequency gives absolute band power.
    return np.trapz(psd[mask], freqs[mask])


def extract_features_from_epoch(epoch_ch_by_time, sfreq):
    """
    Extract temporal and power features from one EEG epoch.

    Parameters
    ----------
    epoch_ch_by_time : numpy.ndarray
        EEG data with shape (n_channels, n_samples).
    sfreq : float
        Sampling frequency of the recording in hertz.

    Returns
    -------
    feature_vector : numpy.ndarray
        One-dimensional array containing the features extracted from the epoch.
    feature_names : list of str
        Names corresponding to the entries in ``feature_vector``.

    Notes
    -----
    The channels are averaged before feature extraction. This produces a
    compact set of global features and avoids multiplying the feature count by
    the number of electrodes. Consequently, spatial differences between
    individual channels are not represented in this feature set.
    """

    # Average across EEG channels to obtain one representative time series
    # for the epoch. The resulting array has shape (n_samples,).
    x = epoch_ch_by_time.mean(axis=0)

    # Time-domain features summarise the central amplitude, variability,
    # extrema and overall signal magnitude of the epoch.
    feat_values = [
        np.mean(x),
        np.std(x),
        np.max(x),
        np.min(x),
        rms(x),
    ]
    feat_names = [
        "mean",
        "std",
        "max",
        "min",
        "rms",
    ]

    # Estimate the power spectral density between 1 and 40 Hz using Welch's
    # method. MNE automatically selects an appropriate segment length when
    # n_per_seg is not specified.
    psd, freqs = psd_array_welch(
        x,
        sfreq=sfreq,
        fmin=1,
        fmax=40,
        verbose=False
    )

    # Integrate the PSD within each predefined frequency band and append the
    # resulting power values to the temporal features.
    for band_name, (fmin, fmax) in bands.items():
        bp = band_power(psd, freqs, fmin, fmax)
        feat_values.append(bp)
        feat_names.append(f"{band_name}_power")

    return np.array(feat_values, dtype=float), feat_names


# -------------------------------------------------------------------------
# Process every epoch file
# -------------------------------------------------------------------------

# Sorting provides a consistent processing order between runs.
epoch_files = sorted(
    glob.glob(os.path.join(EPOCHS_DIR, "*-edf_chin_epochs-epo.fif"))
)

print(f"Found {len(epoch_files)} epoch files.")

for epochs_path in epoch_files:
    print(f"\nProcessing: {epochs_path}")

    # Load all epoch data into memory so that features can be extracted.
    epochs = mne.read_epochs(epochs_path, preload=True, verbose=False)

    # Retain EEG channels only. This excludes channels assigned another type,
    # such as ECG, from the feature calculation.
    # Shape: (n_epochs, n_channels, n_samples)
    X_epochs = epochs.get_data(picks="eeg")

    # The final column of the MNE events array stores the event code.
    # This pipeline assumes: 1 = SWD and 0 = non-SWD.
    y_labels = epochs.events[:, -1].astype(int)

    # Use the sampling frequency stored in the epoch metadata when estimating
    # the PSD rather than hard-coding it.
    sfreq = epochs.info["sfreq"]

    # Extract one feature vector for every epoch.
    X_features = []
    feature_names = None

    for epoch_data in X_epochs:
        feat_vec, feat_names = extract_features_from_epoch(epoch_data, sfreq)
        X_features.append(feat_vec)

        # The same feature order is used for every epoch, so the names only
        # need to be stored once.
        if feature_names is None:
            feature_names = feat_names

    # Convert the list of vectors into a two-dimensional matrix:
    # rows = epochs; columns = extracted features.
    X_features = np.array(X_features)

    # Derive the recording identifier from the input filename so that the
    # corresponding output file can be traced back to its source.
    recording_id = os.path.basename(epochs_path).replace(
        "-edf_chin_epochs-epo.fif", ""
    )

    # Save the feature matrix, class labels and column names together.
    out_path = os.path.join(FEATURES_DIR, f"{recording_id}_features.npz")
    np.savez(
        out_path,
        X=X_features,
        y=y_labels,
        feature_names=np.array(feature_names, dtype=object)
    )

    # Print a concise processing summary for quality control.
    print(f"Saved: {out_path}")
    print(f"  X shape: {X_features.shape}")
    print(f"  y shape: {y_labels.shape}")
    print(
        f"  n SWD: {(y_labels == 1).sum()} | "
        f"n non-SWD: {(y_labels == 0).sum()}"
    )


## 2. Inspect a saved feature file

These checks confirm that the output has the expected structure, that the
number of feature rows matches the number of labels and that both classes are
present. Change `subject_idx` to inspect another recording.


In [ ]:

import os
import glob
import numpy as np

# Locate all feature files and select one recording for inspection.
feature_files = sorted(glob.glob(os.path.join("features", "*_features.npz")))
subject_idx = 0  # Change this index to inspect another feature file.

data = np.load(feature_files[subject_idx], allow_pickle=True)

print("Selected feature file:", feature_files[subject_idx])
print("Stored arrays:", data.files)
print("Feature matrix shape:", data["X"].shape)
print("Label vector shape:", data["y"].shape)
print("Feature names:", data["feature_names"])


In [ ]:

# Assign the saved arrays to shorter variable names for the checks below.
X = data["X"]
y = data["y"]
feature_names = data["feature_names"]

# Boolean comparisons select labels belonging to each class:
# y == 1 selects SWD epochs and y == 0 selects non-SWD epochs.
print("Unique labels:", np.unique(y))
print("n SWD:", np.sum(y == 1))
print("n non-SWD:", np.sum(y == 0))


In [ ]:

# Machine-learning algorithms require finite numerical inputs. These checks
# identify missing values or infinite results that could indicate a problem
# during feature calculation.
print("Any NaNs?", np.isnan(X).any())
print("Any infinities?", np.isinf(X).any())

# Display the observed range of every feature as an additional plausibility
# check and to illustrate why scaling is required before PCA or SVM analysis.
print("Feature minimums:", X.min(axis=0))
print("Feature maximums:", X.max(axis=0))


## 3. Compare SWD and non-SWD feature values

The following summaries are exploratory checks rather than statistical tests.
They allow the direction and magnitude of class differences to be inspected
before classifier training.


In [ ]:

# Split the feature matrix into the two classes using the saved labels.
X_swd = X[y == 1]
X_non = X[y == 0]

# Compare the arithmetic mean of every feature between classes.
for i, name in enumerate(feature_names):
    swd_mean = X_swd[:, i].mean()
    non_mean = X_non[:, i].mean()
    print(f"{name}: SWD={swd_mean:.4e}, non-SWD={non_mean:.4e}")


### Box plots

A separate plot is produced for each feature so that differences in scale do
not obscure the distributions. Amplitude-derived features are reported in
volts, whereas integrated spectral powers are reported in squared volts.


In [ ]:

import matplotlib.pyplot as plt

# More readable labels for presentation in the figures.
display_names = {
    "mean": "Mean",
    "std": "Standard deviation",
    "max": "Maximum",
    "min": "Minimum",
    "rms": "RMS",
    "delta_1_4_power": "Delta power (1–4 Hz)",
    "theta_4_8_power": "Theta power (4–8 Hz)",
    "alpha_8_13_power": "Alpha power (8–13 Hz)",
    "beta_13_30_power": "Beta power (13–30 Hz)",
}

for i, name in enumerate(feature_names):
    plt.figure(figsize=(4, 4))

    # Plot the distributions for the two classes side by side.
    plt.boxplot(
        [X_swd[:, i], X_non[:, i]],
        labels=["SWD", "non-SWD"]
    )

    plt.title(display_names.get(name, name))

    # Temporal features retain amplitude units. Spectral features represent
    # integrated power and therefore use squared amplitude units.
    if name in ["mean", "std", "max", "min", "rms"]:
        plt.ylabel("Amplitude (V)")
    else:
        plt.ylabel("Band power (V²)")

    plt.tight_layout()
    plt.show()


## 4. Visualise class separation using PCA

The features have different numerical scales, so they are standardised before
PCA. PCA reduces the feature matrix to two components for visualisation.

This is an exploratory step only. The PCA coordinates generated here are not
used as inputs to the SVM classifier.


In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np


# Select one feature file. Use the same index as above when comparing the PCA
# plot with the corresponding feature summaries and box plots.
subject_idx = 0

data = np.load(feature_files[subject_idx], allow_pickle=True)
X = data["X"]
y = data["y"]

print("Recording:", os.path.basename(feature_files[subject_idx]))
print("Shape:", X.shape)


# Standardise each feature to zero mean and unit variance. This prevents
# high-magnitude variables, particularly spectral power, from dominating PCA
# solely because of their numerical scale.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# Reduce the standardised feature matrix to two principal components.
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance:", pca.explained_variance_ratio_)
print("Total variance:", pca.explained_variance_ratio_.sum())


# Plot each epoch in the two-dimensional PCA space, coloured by class.
plt.figure(figsize=(7, 6))

plt.scatter(
    X_pca[y == 0, 0],
    X_pca[y == 0, 1],
    s=60,
    alpha=0.8,
    label="non-SWD"
)

plt.scatter(
    X_pca[y == 1, 0],
    X_pca[y == 1, 1],
    s=60,
    alpha=0.8,
    label="SWD"
)

plt.xlabel(
    f"PC1 ({100 * pca.explained_variance_ratio_[0]:.1f}% variance)"
)
plt.ylabel(
    f"PC2 ({100 * pca.explained_variance_ratio_[1]:.1f}% variance)"
)
plt.title(f"PCA of {os.path.basename(feature_files[subject_idx])}")

plt.legend()
plt.tight_layout()
plt.show()


## 5. Create a tabular class summary

This table provides the mean value of every feature for the selected recording.
It is another compact quality-control view of the same class comparison shown
above.


In [ ]:

import pandas as pd

# Recreate the class-specific arrays in case the PCA cell was run independently.
X_swd = X[y == 1]
X_non = X[y == 0]

summary = []

for i, name in enumerate(feature_names):
    summary.append({
        "feature": name,
        "SWD_mean": X_swd[:, i].mean(),
        "nonSWD_mean": X_non[:, i].mean()
    })

summary_df = pd.DataFrame(summary)
print(summary_df)


## 6. Inspect additional recordings

Change `subject_idx` below to inspect any other feature file using the same
checks. This replaces the repeated file-specific cells in the working notebook
and reduces the risk of loading one file while accidentally printing the name
of another.


In [ ]:

# Display the available files and their indices.
for idx, path in enumerate(feature_files):
    print(idx, os.path.basename(path))


In [ ]:

# Select another recording by changing this index.
subject_idx = 1

data = np.load(feature_files[subject_idx], allow_pickle=True)

X = data["X"]
y = data["y"]
feature_names = data["feature_names"]

X_swd = X[y == 1]
X_non = X[y == 0]

print("\nSelected feature file:", os.path.basename(feature_files[subject_idx]))
print("Stored arrays:", data.files)
print("Feature matrix shape:", X.shape)
print("Label vector shape:", y.shape)
print("Unique labels:", np.unique(y))
print("n SWD:", np.sum(y == 1))
print("n non-SWD:", np.sum(y == 0))

summary = []
for i, name in enumerate(feature_names):
    summary.append({
        "feature": name,
        "SWD_mean": X_swd[:, i].mean(),
        "nonSWD_mean": X_non[:, i].mean()
    })

summary_df = pd.DataFrame(summary)
print("\nFeature means by class:")
print(summary_df)


### The final feature set contains nine variables:

- Mean
- Standard deviation
- Maximum
- Minimum
- RMS
- Delta power
- Theta power
- Alpha power
- Beta power
